# 02 — Sentence-Transformer Embeddings

Encode every extracted resume into a dense vector using `all-MiniLM-L6-v2` (384-dim).

**Input:** `data/processed/resumes_extracted.json` (from notebook 01)  
**Outputs:**
- `data/processed/embeddings.npy` — (N, 384) float32 array
- `data/processed/resumes_with_embeddings.json` — metadata aligned by index with the .npy rows

In [1]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

PROJECT_ROOT = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH  = PROCESSED_DIR / "resumes_extracted.json"
EMB_PATH    = PROCESSED_DIR / "embeddings.npy"
META_PATH   = PROCESSED_DIR / "resumes_with_embeddings.json"

MODEL_NAME  = "all-MiniLM-L6-v2"
BATCH_SIZE  = 32

print(f"Input  : {INPUT_PATH}")
print(f"Outputs: {EMB_PATH}")
print(f"         {META_PATH}")
print(f"Model  : {MODEL_NAME}")
print(f"Device : {'cuda' if torch.cuda.is_available() else 'cpu'}")

Input  : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/resumes_extracted.json
Outputs: /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/embeddings.npy
         /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/resumes_with_embeddings.json
Model  : all-MiniLM-L6-v2
Device : cpu


## 1 — Load extracted resumes

In [2]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} resumes")

source_counts = {}
for r in records:
    source_counts[r["source"]] = source_counts.get(r["source"], 0) + 1
for src, cnt in sorted(source_counts.items()):
    print(f"  {src:20s} : {cnt}")

Loaded 807 resumes
  ds3_board            : 46
  ds3_members          : 274
  kaggle               : 357
  train                : 130


## 2 — Load the model

In [3]:
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


## 3 — Encode all resumes

We embed the full cleaned text of each resume. For very long resumes the model
has a 256-token window by default — `sentence-transformers` will silently
truncate. This is usually fine for ranking because the most information-dense
portion of a resume (skills, titles) tends to appear near the top.

In [4]:
texts = [r["text"] for r in records]

print(f"Encoding {len(texts)} resumes (batch_size={BATCH_SIZE}) ...")
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2-normalize so dot product == cosine similarity
)

embeddings = np.array(embeddings, dtype=np.float32)
print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Norm of first vector (should be ~1.0): {np.linalg.norm(embeddings[0]):.4f}")

Encoding 807 resumes (batch_size=32) ...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]


Embeddings shape: (807, 384)
Norm of first vector (should be ~1.0): 1.0000


## 4 — Save embeddings & aligned metadata

In [5]:
np.save(EMB_PATH, embeddings)
print(f"Saved embeddings: {EMB_PATH}  ({EMB_PATH.stat().st_size / 1024:.1f} KB)")

# The metadata file mirrors the row order of the .npy exactly.
# We strip the full text to keep the file size small — the text
# is still available in resumes_extracted.json if needed.
meta_records = []
for r in records:
    meta_records.append({
        "filename": r["filename"],
        "source": r["source"],
        "file_path": r["file_path"],
        "word_count": r["word_count"],
        "text_preview": r["text"][:400],
        "metadata": r.get("metadata", {}),
    })

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_records, f, indent=2, ensure_ascii=False)

print(f"Saved metadata  : {META_PATH}  ({META_PATH.stat().st_size / 1024:.1f} KB)")

Saved embeddings: /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/embeddings.npy  (1210.6 KB)
Saved metadata  : /Users/guest2/Desktop/repos/talentlens/TalentLens_Public/data/processed/resumes_with_embeddings.json  (600.2 KB)


## 5 — Quick inspection

Sanity-check: pick a resume and find its nearest neighbours in embedding space.

In [6]:
query_idx = 0
sims = embeddings @ embeddings[query_idx]  # cosine similarities (already normalized)

top_k = 5
top_indices = np.argsort(sims)[::-1][:top_k + 1]  # +1 because the resume matches itself

print(f"Nearest neighbours of: {records[query_idx]['filename']}")
print(f"{'Rank':<6} {'Score':<8} {'Source':<15} {'Filename'}")
print("-" * 60)
for rank, idx in enumerate(top_indices):
    print(f"{rank:<6} {sims[idx]:<8.4f} {records[idx]['source']:<15} {records[idx]['filename']}")

Nearest neighbours of: member_resume_aaditya_khanuja.pdf
Rank   Score    Source          Filename
------------------------------------------------------------
0      1.0000   ds3_members     member_resume_aaditya_khanuja.pdf
1      0.6928   ds3_members     member_resume_wanhan_sun.pdf
2      0.6896   ds3_board       board_resume_36.pdf
3      0.6859   ds3_members     member_resume_praveen_sharma.pdf
4      0.6834   ds3_members     member_resume_hanit_kumar.pdf
5      0.6826   ds3_members     member_resume_nikhita_kadam.pdf


## 6 — (Optional) Visualize embedding clusters

Reduce to 2D with UMAP or t-SNE and colour by source to see if the different
resume pools separate naturally.

In [7]:
# TODO: Uncomment and run once you have enough resumes to make this interesting.
#
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt
#
# tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings) - 1))
# coords = tsne.fit_transform(embeddings)
#
# sources = [r["source"] for r in records]
# unique_sources = sorted(set(sources))
# colors = {s: c for s, c in zip(unique_sources, plt.cm.tab10.colors)}
#
# plt.figure(figsize=(10, 8))
# for src in unique_sources:
#     mask = [i for i, s in enumerate(sources) if s == src]
#     plt.scatter(coords[mask, 0], coords[mask, 1], label=src, alpha=0.6, s=20)
# plt.legend()
# plt.title("Resume embeddings (t-SNE)")
# plt.tight_layout()
# plt.show()